# Working with Canadian open source data

In this notebook, I resesarched Canadian data sources and attempted to perform the same analysis as shown in overLayingData.ipynb. However, I was unable to find suitable information on Canadian transmission lines, so this remains unfinished. 

In [ ]:
import geopandas as gp
import pandas as pd
'''import matplotlib.pyplot as plt
import contextily as cx
import osmnx as ox'''

In [ ]:
#Data about Candian emission sources is read from a CSV file and made into a pandas dataframe
canadaImport = pd.read_csv('Data/PDGES-GHGRP-GHGEmissionsSourcesGES-2022.csv')
#The data is cleaned - several points in the data set had incorrect longitude values and those are corrected
canadaImport = canadaImport.drop_duplicates()
for index, row in canadaImport.iterrows():
    currValue = row['Longitude']
    if (currValue > 0):
        negativeValue = currValue * -1
        canadaImport['Longitude'].replace(currValue, negativeValue, inplace = True)
#The dataframe is converted into a geodataframe with the coordinate reference system ESPG:4326
canadaEmissions = gp.GeoDataFrame(
    canadaImport, geometry=gp.points_from_xy(canadaImport.Longitude, canadaImport.Latitude), crs = 'EPSG:4326'
)
#The geodataframe is plotted with varying colors depending on each emission source's CO2 emissions. 
canadaEmissions.plot(column="CO2 (tonnes)", legend=True, cmap = "autumn")

In [ ]:
#This creates a map showing a 2.3 mile buffer around the transmission lines shown in the canada_transmission_lines.geojson data, but that data is incomplete so the map is limited. 
canadaPowerLines = gp.read_file('Data/canada_transmission_lines.geojson')
canadaLineBuffers = canadaPowerLines.buffer(0.043, resolution=16, cap_style='round', join_style='round', mitre_limit=5.0, single_sided=False)
canadaLineBuffers.to_crs('EPSG:4326')
canadaBuffersGdf = gp.GeoDataFrame(canadaLineBuffers, geometry=gp.GeoSeries(canadaLineBuffers))
canadaBuffersGdf.plot()